# End-to-End Automated Fabric Thread Density Counter (1.0 cm)

**The mechanistic justification for this step is:** Provide an integrated, production-ready computer vision pipeline that takes a raw macro fabric photograph and spatial calibration scale, automatically crops a 1.0 cm² ROI, rectifies angular skew, executes dual-engine frequency and spatial thread counting, performs consensus validation, and calculates physical yarn spacing with an objective confidence score.

In [1]:
# The mechanistic justification for this step is: configure the global Kaggle pathing invariant with local fallback routing and initialize PyTorch execution on GPU hardware.
import os
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

INPUT_DIR = "/kaggle/input/fabric-data" if os.path.exists("/kaggle/input/fabric-data") else "data"
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "results"
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(os.path.join(INPUT_DIR, "synthetic"), exist_ok=True)
os.makedirs(os.path.join(INPUT_DIR, "samples"), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory (MB): {torch.cuda.memory_allocated() / (1024 * 1024):.2f}")
print(f"Input Directory: {INPUT_DIR}")
print(f"Working Directory: {WORKING_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Allocated Memory (MB): 0.00
Input Directory: data
Working Directory: results


In [2]:
# The mechanistic justification for this step is: generate a high-resolution macro textile capture (800x800 px, 400 px/cm = 2.0 cm x 2.0 cm) with +12.0 deg tilt, non-uniform spatial illumination gradient, and known density (Warp: 36 ends/cm, Weft: 30 picks/cm) to benchmark the full end-to-end extraction pipeline.
swatch_size = 800
pixels_per_cm = 400
true_warp = 36
true_weft = 30
true_skew_deg = 12.0

x = np.linspace(0, 2.0, swatch_size, endpoint=False)
y = np.linspace(0, 2.0, swatch_size, endpoint=False)
X, Y = np.meshgrid(x, y)

warp_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_warp * X))
weft_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_weft * Y))
interlace = np.sin(2.0 * np.pi * true_warp * X) * np.sin(2.0 * np.pi * true_weft * Y)
base_texture = 0.45 * warp_signal + 0.45 * weft_signal + 0.1 * interlace

# Non-uniform lighting gradient (vignetting + linear gradient across field of view)
illumination = 0.8 + 0.2 * np.cos(np.pi * (X - 1.0)) * np.cos(np.pi * (Y - 1.0))
noisy_swatch = np.clip(base_texture * illumination + np.random.normal(0, 0.04, (swatch_size, swatch_size)), 0.0, 1.0)

raw_pil = Image.fromarray((noisy_swatch * 255).astype(np.uint8))
skewed_swatch_pil = raw_pil.rotate(true_skew_deg, resample=Image.Resampling.BICUBIC)
swatch_path = os.path.join(INPUT_DIR, "synthetic", "macro_swatch_skewed_12deg.png")
skewed_swatch_pil.save(swatch_path)

skewed_swatch = np.array(skewed_swatch_pil, dtype=np.float32) / 255.0
print(f"Swatch Dimensions: {skewed_swatch.shape}")
print(f"Spatial Resolution (px/cm): {pixels_per_cm}")
print(f"Pixel Pitch (um/px): {10000.0 / pixels_per_cm:.2f}")
print(f"True Warp (ends/cm): {true_warp}")
print(f"True Weft (picks/cm): {true_weft}")
print(f"True Skew Angle (deg): {true_skew_deg}")

Swatch Dimensions: (800, 800)
Spatial Resolution (px/cm): 400
Pixel Pitch (um/px): 25.00
True Warp (ends/cm): 36
True Weft (picks/cm): 30
True Skew Angle (deg): 12.0


In [3]:
# The mechanistic justification for this step is: convert the spatial calibration scale into physical pixel dimensions and extract a centered 1.0 cm x 1.0 cm Region of Interest (ROI) from the macro capture.
roi_dim = pixels_per_cm # 400 px = 1.0 cm
h, w = skewed_swatch.shape
cx, cy = w // 2, h // 2
half_roi = roi_dim // 2

roi_raw = skewed_swatch[cy - half_roi : cy + half_roi, cx - half_roi : cx + half_roi]
roi_path = os.path.join(WORKING_DIR, "roi_1cm_raw.png")
Image.fromarray((roi_raw * 255).astype(np.uint8)).save(roi_path)

print(f"ROI Dimensions: {roi_raw.shape}")
print(f"ROI Field of View: 1.0 cm x 1.0 cm ({roi_raw.shape[0]}x{roi_raw.shape[1]} pixels)")
print(f"Sampling Pitch: {10000.0 / roi_dim:.2f} um/pixel")

ROI Dimensions: (400, 400)
ROI Field of View: 1.0 cm x 1.0 cm (400x400 pixels)
Sampling Pitch: 25.00 um/pixel


In [4]:
# The mechanistic justification for this step is: apply Contrast Limited Adaptive Histogram Equalization (CLAHE) to suppress field illumination gradients, then calculate 2D FFT Polar Radon energy integrals across [-45 deg, +45 deg] to isolate fabric orientation.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
roi_uint8 = (roi_raw * 255).astype(np.uint8)
roi_clahe = clahe.apply(roi_uint8).astype(np.float32) / 255.0

tensor_roi = torch.tensor(roi_clahe, device=device, dtype=torch.float32)
f_input = tensor_roi - torch.mean(tensor_roi)
fft2 = torch.fft.fft2(f_input)
fft_shift = torch.fft.fftshift(fft2)
magnitude = torch.abs(fft_shift)

rows, cols = magnitude.shape
crow, ccol = rows // 2, cols // 2

angles = np.linspace(-45, 45, 181)
angular_energies = []
r_vals = torch.arange(15, int(min(rows, cols) * 0.45), device=device, dtype=torch.float32)

for theta in angles:
    rad = np.deg2rad(theta)
    u_idx = torch.clamp(torch.round(crow - r_vals * np.sin(rad)).long(), 0, rows - 1)
    v_idx = torch.clamp(torch.round(ccol + r_vals * np.cos(rad)).long(), 0, cols - 1)
    energy = torch.sum(magnitude[u_idx, v_idx]).item()
    angular_energies.append(energy)

angular_energies = np.array(angular_energies)
detected_skew_deg = float(angles[np.argmax(angular_energies)])
skew_error_deg = abs(detected_skew_deg - true_skew_deg)

print(f"Detected Skew Angle (deg): {detected_skew_deg}")
print(f"Ground Truth Skew Angle (deg): {true_skew_deg}")
print(f"Skew Estimation Delta (deg): {skew_error_deg}")

Detected Skew Angle (deg): 11.5
Ground Truth Skew Angle (deg): 12.0
Skew Estimation Delta (deg): 0.5


In [5]:
# The mechanistic justification for this step is: execute inverse affine rotation by -theta_detected to restore orthogonal yarn orientation and crop center 70% to eliminate rotation zero-padding boundaries.
roi_pil = Image.fromarray((roi_clahe * 255).astype(np.uint8))
rectified_pil = roi_pil.rotate(-detected_skew_deg, resample=Image.Resampling.BICUBIC)
rectified_roi = np.array(rectified_pil, dtype=np.float32) / 255.0

margin = int(roi_dim * 0.15)
crop_roi = rectified_roi[margin : roi_dim - margin, margin : roi_dim - margin]
crop_dim = crop_roi.shape[0]
crop_scale = crop_dim / roi_dim
crop_length_cm = crop_dim / pixels_per_cm

print(f"Raw ROI Shape: {roi_raw.shape}")
print(f"Rectified Crop Shape: {crop_roi.shape}")
print(f"Crop Scale Factor: {crop_scale:.4f}")
print(f"Crop Physical Length: {crop_length_cm:.4f} cm")

Raw ROI Shape: (400, 400)
Rectified Crop Shape: (280, 280)
Crop Scale Factor: 0.7000
Crop Physical Length: 0.7000 cm


In [6]:
# The mechanistic justification for this step is: extract fundamental spatial frequency harmonics along orthogonal axes from the rectified 2D FFT magnitude spectrum and apply quadratic parabolic interpolation to achieve sub-pixel yarn counting.
crop_tensor = torch.tensor(crop_roi, device=device, dtype=torch.float32)
crop_f = crop_tensor - torch.mean(crop_tensor)
crop_mag = torch.abs(torch.fft.fftshift(torch.fft.fft2(crop_f))).cpu().numpy()

rcrow, rccol = crop_mag.shape[0] // 2, crop_mag.shape[1] // 2
max_bins = int(crop_mag.shape[1] * 0.45)

horiz_spec = np.mean(crop_mag[rcrow - 2 : rcrow + 3, rccol + 3 : rccol + max_bins], axis=0)
vert_spec = np.mean(crop_mag[rcrow + 3 : rcrow + max_bins, rccol - 2 : rccol + 3], axis=1)

def refine_peak(spec, peak_idx):
    if 0 < peak_idx < len(spec) - 1:
        alpha, beta, gamma = spec[peak_idx - 1], spec[peak_idx], spec[peak_idx + 1]
        denom = alpha - 2 * beta + gamma
        if abs(denom) > 1e-8:
            return float(peak_idx) + 0.5 * (alpha - gamma) / denom
    return float(peak_idx)

h_idx = np.argmax(horiz_spec)
v_idx = np.argmax(vert_spec)

pred_warp_fft = float(round((refine_peak(horiz_spec, h_idx) + 3) / crop_scale, 1))
pred_weft_fft = float(round((refine_peak(vert_spec, v_idx) + 3) / crop_scale, 1))

# Spectral peak-to-noise ratio
spnr_warp = float(horiz_spec[h_idx] / (np.median(horiz_spec) + 1e-8))
spnr_weft = float(vert_spec[v_idx] / (np.median(vert_spec) + 1e-8))

print(f"Engine 1 (2D FFT) Pred Warp (ends/cm): {pred_warp_fft} (True: {true_warp}) | Delta: {abs(pred_warp_fft - true_warp):.2f}")
print(f"Engine 1 (2D FFT) Pred Weft (picks/cm): {pred_weft_fft} (True: {true_weft}) | Delta: {abs(pred_weft_fft - true_weft):.2f}")
print(f"Warp SPNR: {spnr_warp:.2f} | Weft SPNR: {spnr_weft:.2f}")

Engine 1 (2D FFT) Pred Warp (ends/cm): 35.79999923706055 (True: 36) | Delta: 0.20
Engine 1 (2D FFT) Pred Weft (picks/cm): 30.0 (True: 30) | Delta: 0.00
Warp SPNR: 147.47 | Weft SPNR: 266.47


In [7]:
# The mechanistic justification for this step is: compute marginal projection profiles across columns (warp) and rows (weft), extract peaks with minimum-distance and prominence constraints, and scale by crop length to determine physical thread density.
warp_prof = np.mean(crop_roi, axis=0)
weft_prof = np.mean(crop_roi, axis=1)

warp_contrast = float(np.std(warp_prof))
weft_contrast = float(np.std(weft_prof))

def find_peaks_1d(signal, min_dist=3, prominence=0.02):
    n = len(signal)
    peaks = []
    for i in range(1, n - 1):
        if signal[i] > signal[i-1] and signal[i] > signal[i+1]:
            left_min = np.min(signal[max(0, i - min_dist) : i])
            right_min = np.min(signal[i + 1 : min(n, i + min_dist + 1)])
            prom = signal[i] - max(left_min, right_min)
            if prom >= prominence:
                if len(peaks) == 0 or (i - peaks[-1]) >= min_dist:
                    peaks.append(i)
                elif signal[i] > signal[peaks[-1]]:
                    peaks[-1] = i
    return np.array(peaks)

min_d = max(3, int(pixels_per_cm / 80)) # Dynamic minimum distance
warp_peaks = find_peaks_1d(warp_prof, min_dist=min_d, prominence=0.015)
weft_peaks = find_peaks_1d(weft_prof, min_dist=min_d, prominence=0.015)

pred_warp_proj = int(round(len(warp_peaks) / crop_length_cm))
pred_weft_proj = int(round(len(weft_peaks) / crop_length_cm))

# Physical spacing
warp_spacings_px = np.diff(warp_peaks) if len(warp_peaks) > 1 else np.array([roi_dim / pred_warp_proj])
weft_spacings_px = np.diff(weft_peaks) if len(weft_peaks) > 1 else np.array([roi_dim / pred_weft_proj])

mean_warp_pitch_um = float(np.mean(warp_spacings_px) * (10000.0 / pixels_per_cm))
mean_weft_pitch_um = float(np.mean(weft_spacings_px) * (10000.0 / pixels_per_cm))

print(f"Engine 2 (Projection) Pred Warp (ends/cm): {pred_warp_proj} (Peaks: {len(warp_peaks)}) | Delta: {abs(pred_warp_proj - true_warp)}")
print(f"Engine 2 (Projection) Pred Weft (picks/cm): {pred_weft_proj} (Peaks: {len(weft_peaks)}) | Delta: {abs(pred_weft_proj - true_weft)}")
print(f"Warp Mean Spacing: {mean_warp_pitch_um:.2f} um | Weft Mean Spacing: {mean_weft_pitch_um:.2f} um")

Engine 2 (Projection) Pred Warp (ends/cm): 36 (Peaks: 25) | Delta: 0
Engine 2 (Projection) Pred Weft (picks/cm): 30 (Peaks: 21) | Delta: 0
Warp Mean Spacing: 277.08 um | Weft Mean Spacing: 332.50 um


In [8]:
# The mechanistic justification for this step is: validate consensus between the frequency-domain and spatial-domain engines, synthesize the final thread density, and calculate a multi-attribute quality score Q in [0, 100%].
delta_warp = abs(pred_warp_fft - pred_warp_proj)
delta_weft = abs(pred_weft_fft - pred_weft_proj)

consensus_warp = int(round((pred_warp_fft + pred_warp_proj) / 2.0))
consensus_weft = int(round((pred_weft_fft + pred_weft_proj) / 2.0))
is_consistent = (delta_warp <= 2.0) and (delta_weft <= 2.0)

# Multi-attribute Quality Metric Q
s1_spectral = min(1.0, (spnr_warp + spnr_weft) / 10.0)
s2_contrast = min(1.0, (warp_contrast + weft_contrast) / 0.20)
s3_agreement = max(0.0, 1.0 - (delta_warp + delta_weft) / 4.0)
quality_score_pct = float(round((0.35 * s1_spectral + 0.35 * s2_contrast + 0.30 * s3_agreement) * 100.0, 1))

print(f"Consensus Warp (ends/cm): {consensus_warp} (True: {true_warp})")
print(f"Consensus Weft (picks/cm): {consensus_weft} (True: {true_weft})")
print(f"Inter-Engine Delta (Warp, Weft): ({delta_warp:.2f}, {delta_weft:.2f})")
print(f"Consensus Validated: {is_consistent}")
print(f"Confidence Quality Score Q (%): {quality_score_pct}")

Consensus Warp (ends/cm): 36 (True: 36)
Consensus Weft (picks/cm): 30 (True: 30)
Inter-Engine Delta (Warp, Weft): (0.20, 0.00)
Consensus Validated: True
Confidence Quality Score Q (%): 98.5


In [9]:
# The mechanistic justification for this step is: render a comprehensive 6-panel production report visualizing ROI cropping, skew detection, rectified texture, marginal projections with peak markers, and harmonic spectra, and save the artifact.
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# 1. Macro Swatch with ROI overlay
axes[0, 0].imshow(skewed_swatch, cmap='gray')
rect_patch = plt.Rectangle((cx - half_roi, cy - half_roi), roi_dim, roi_dim, edgecolor='red', facecolor='none', lw=2)
axes[0, 0].add_patch(rect_patch)
axes[0, 0].set_title(f'Raw Swatch (800x800 px) + 1cm ROI Box', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

# 2. Polar Radon Energy Scan
axes[0, 1].plot(angles, angular_energies, color='darkblue', lw=2)
axes[0, 1].axvline(detected_skew_deg, color='red', linestyle='--', label=f'Peak @ {detected_skew_deg} deg')
axes[0, 1].set_title(f'Polar Radon Orientation Sweep (Err: {skew_error_deg:.2f} deg)', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Angle (degrees)')
axes[0, 1].set_ylabel('Spectral Energy')
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, alpha=0.3)

# 3. Rectified Fabric Crop
axes[0, 2].imshow(crop_roi, cmap='gray')
axes[0, 2].set_title(f'Rectified Crop (0.7cm FOV, -{detected_skew_deg} deg)', fontsize=11, fontweight='bold')
axes[0, 2].axis('off')

# 4. Warp Marginal Projection
axes[1, 0].plot(warp_prof, color='navy', lw=1.5, label='Warp Profile')
axes[1, 0].plot(warp_peaks, warp_prof[warp_peaks], 'rx', markersize=6, label=f'Peaks: {len(warp_peaks)} ({pred_warp_proj} ends/cm)')
axes[1, 0].set_title(f'Warp Projection: {consensus_warp} ends/cm (Pitch: {mean_warp_pitch_um:.1f} um)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Pixel Index (Crop X)')
axes[1, 0].set_ylabel('Intensity')
axes[1, 0].legend(loc='upper right')
axes[1, 0].grid(True, alpha=0.3)

# 5. Weft Marginal Projection
axes[1, 1].plot(weft_prof, color='darkgreen', lw=1.5, label='Weft Profile')
axes[1, 1].plot(weft_peaks, weft_prof[weft_peaks], 'rx', markersize=6, label=f'Peaks: {len(weft_peaks)} ({pred_weft_proj} picks/cm)')
axes[1, 1].set_title(f'Weft Projection: {consensus_weft} picks/cm (Pitch: {mean_weft_pitch_um:.1f} um)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Pixel Index (Crop Y)')
axes[1, 1].set_ylabel('Intensity')
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

# 6. Rectified 2D FFT Spectrum
axes[1, 2].imshow(np.log1p(crop_mag), cmap='inferno')
axes[1, 2].set_title(f'Rectified 2D FFT Spectrum (Quality: {quality_score_pct}%)', fontsize=11, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
report_path = os.path.join(WORKING_DIR, 'end_to_end_analysis_report.png')
plt.savefig(report_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Production Report Saved: {report_path}")

Production Report Saved: results/end_to_end_analysis_report.png


In [10]:
# The mechanistic justification for this step is: persist the complete end-to-end production analysis payload to results/output.json and results/output.txt for central provenance auditability.
final_results = {
    "calibration": {
        "pixels_per_cm": pixels_per_cm,
        "pixel_pitch_um": round(10000.0 / pixels_per_cm, 2),
        "roi_field_of_view_cm": [1.0, 1.0],
        "roi_pixel_dimensions": [roi_dim, roi_dim]
    },
    "ground_truth": {
        "warp_ends_per_cm": true_warp,
        "weft_picks_per_cm": true_weft,
        "skew_angle_deg": true_skew_deg
    },
    "orientation_engine": {
        "detected_skew_deg": detected_skew_deg,
        "skew_error_deg": skew_error_deg,
        "rectification_status": "orthogonal_aligned"
    },
    "spectral_fft_engine": {
        "warp_ends_per_cm": pred_warp_fft,
        "weft_picks_per_cm": pred_weft_fft,
        "warp_spnr": round(spnr_warp, 2),
        "weft_spnr": round(spnr_weft, 2),
        "delta_warp": round(abs(pred_warp_fft - true_warp), 2),
        "delta_weft": round(abs(pred_weft_fft - true_weft), 2)
    },
    "spatial_projection_engine": {
        "warp_ends_per_cm": pred_warp_proj,
        "weft_picks_per_cm": pred_weft_proj,
        "mean_warp_pitch_um": round(mean_warp_pitch_um, 2),
        "mean_weft_pitch_um": round(mean_weft_pitch_um, 2),
        "delta_warp": abs(pred_warp_proj - true_warp),
        "delta_weft": abs(pred_weft_proj - true_weft)
    },
    "consensus_validation": {
        "validated_warp_ends_per_cm": consensus_warp,
        "validated_weft_picks_per_cm": consensus_weft,
        "warp_discrepancy": round(delta_warp, 2),
        "weft_discrepancy": round(delta_weft, 2),
        "is_consistent": is_consistent,
        "confidence_quality_score_pct": quality_score_pct
    }
}

with open(os.path.join(WORKING_DIR, "output.json"), "w") as f:
    json.dump(final_results, f, indent=2)

with open(os.path.join(WORKING_DIR, "output.txt"), "w") as f:
    f.write("=== PRODUCTION FABRIC ANALYSIS REPORT (1.0 CM) ===\n")
    f.write(f"Spatial Calibration: {pixels_per_cm} px/cm | Pitch: {10000.0/pixels_per_cm:.2f} um/px\n")
    f.write(f"Orientation Rectification: Detected {detected_skew_deg} deg (True {true_skew_deg} deg) | Error: {skew_error_deg:.2f} deg\n")
    f.write(f"Validated Thread Density: {consensus_warp} Ends/cm (Warp) | {consensus_weft} Picks/cm (Weft)\n")
    f.write(f"Physical Pitch: Warp = {mean_warp_pitch_um:.1f} um | Weft = {mean_weft_pitch_um:.1f} um\n")
    f.write(f"Dual-Engine Discrepancy: Warp Delta = {delta_warp:.2f} | Weft Delta = {delta_weft:.2f}\n")
    f.write(f"Confidence Quality Score: {quality_score_pct}% | Consensus Verified: {is_consistent}\n")

print(json.dumps(final_results, indent=2))

{
  "calibration": {
    "pixels_per_cm": 400,
    "pixel_pitch_um": 25.0,
    "roi_field_of_view_cm": [
      1.0,
      1.0
    ],
    "roi_pixel_dimensions": [
      400,
      400
    ]
  },
  "ground_truth": {
    "warp_ends_per_cm": 36,
    "weft_picks_per_cm": 30,
    "skew_angle_deg": 12.0
  },
  "orientation_engine": {
    "detected_skew_deg": 11.5,
    "skew_error_deg": 0.5,
    "rectification_status": "orthogonal_aligned"
  },
  "spectral_fft_engine": {
    "warp_ends_per_cm": 35.79999923706055,
    "weft_picks_per_cm": 30.0,
    "warp_spnr": 147.47,
    "weft_spnr": 266.47,
    "delta_warp": 0.2,
    "delta_weft": 0.0
  },
  "spatial_projection_engine": {
    "warp_ends_per_cm": 36,
    "weft_picks_per_cm": 30,
    "mean_warp_pitch_um": 277.08,
    "mean_weft_pitch_um": 332.5,
    "delta_warp": 0,
    "delta_weft": 0
  },
  "consensus_validation": {
    "validated_warp_ends_per_cm": 36,
    "validated_weft_picks_per_cm": 30,
    "warp_discrepancy": 0.2,
    "weft_discrepanc